# Pipeline to streamline the following tasks

 1. Extract YT video urls to create a curated dataset  
 >[VIDEO_ID, VIDEO_TITLE, VIDEO_URL, CHANNEL, DURATION_SECS,
UPLOAD_DATE, LANGUAGE, VIEW_COUNT, EXTRACTION_TIMESTAMP,
AUDIO_PATH, TRANSCRIPT_PATH, TRANSCRIPT_STATUS]  

 [2]. Extract audios from the videos using URLS or video_IDs.
 >Strip the .mp3 codec audio files from the videos and save them as a dataset.
 >> DataSet > VideoID > [audio.mp3]

 3. Establish process to extract / generate transcriptions from the videos.
 >Likely use of AI models for transcription due to no presence of youtube generated transcripts in the videos.

## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Modules

In [ ]:
%%capture
!pip install yt-dlp -U
# !apt-get update
!apt-get install -y ffmpeg

import pandas as pd
import yt_dlp

## Load and Filter DF

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/AnnamAI Tasks/YT_DATASET.csv')
print(f"Initial number of rows: {len(df)}")

def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID

def YT_VIDEO_ID_generator(VIDEO_LINK:str) -> str:
    """Generator function to generate VIDEO_IDS from Link"""
    return VIDEO_LINK[-11:]

def exclude_shorts_and_Large(df):
    # omit videos which are less than 2 minutes (120 seconds) or larger than 2000 seconds
    condition = (df['Duration'] >= 120) & (df['Duration'] <= 2000)
    filtered_df = df[condition]
    residue_df = df[~condition]
    return filtered_df, residue_df

df_final, duration_residue_df = exclude_shorts_and_Large(df)
print(f"Rows after duration filtering: {len(df_final)}")
print(f"Rows removed by duration filtering: {len(duration_residue_df)}")

df = df_final

duration_residue_df.to_csv('/content/residue.csv', index=False)
print(f"Total rows in residue.csv: {len(duration_residue_df)}")


Initial number of rows: 13995
Rows after duration filtering: 12416
Rows removed by duration filtering: 1579
Total rows in residue.csv: 1579


In [ ]:
df.head(10)

,YT_VIDEO_ID,YT_VIDEO_TITLE,YT_VIDEO_LINK,Duration,Channel
0,SxgaGXcQ8ZY,Krishi Darshan एकिकृत कृषि प्रणाली,https://www.youtube.com/watch?v=SxgaGXcQ8ZY,1377.0,Doordarshan National
1,JLwhLr4ZVd0,Krishi Darshan Bhagwani ke vikas ka harayana...,https://www.youtube.com/watch?v=JLwhLr4ZVd0,1472.0,Doordarshan National
2,ET8rwerJTg0,KRISHI DARSHAN GAON SAMRIDH BHART SAMRIDH,https://www.youtube.com/watch?v=ET8rwerJTg0,1491.0,Doordarshan National
3,BfMmAW-58ng,Krishi Darshan Chana Fasal,https://www.youtube.com/watch?v=BfMmAW-58ng,1221.0,Doordarshan National
4,VzSq2eW8ZG8,Krishi Darshan Zaed Phasal new,https://www.youtube.com/watch?v=VzSq2eW8ZG8,1476.0,Doordarshan National
5,tqu_7SftoxM,Suksham Sichai Se Badhyae Utpadan,https://www.youtube.com/watch?v=tqu_7SftoxM,1438.0,Doordarshan National
6,wlwZIC62hXQ,Gehu ki buaai,https://www.youtube.com/watch?v=wlwZIC62hXQ,1462.0,Doordarshan National
7,k3mJBSHc4m4,Krishi Darshan जायद की फसलों का कीटों से बचाव,https://www.youtube.com/watch?v=k3mJBSHc4m4,1472.0,Doordarshan National
8,_CrkOwCc4pU,"krishi darshan जैविक कृषि बाजार, दिल्ली हाट",https://www.youtube.com/watch?v=_CrkOwCc4pU,1538.0,Doordarshan National
9,5WlTrocLyL4,Sabjiyo mein kit prabandhan,https://www.youtube.com/watch?v=5WlTrocLyL4,1652.0,Doordarshan National


In [ ]:
data_lists = {col: df[col].tolist() for col in df.columns}

# To verify, you can print the first few items of each list
for col_name, col_list in data_lists.items():
    print(f"Column '{col_name}': {col_list[:5]}...")

# Create separate list variables for each column from the data_lists dictionary
for col_name, col_list in data_lists.items():
    # Sanitize column names to be valid Python variable names and add '_list' suffix
    variable_name = col_name
    globals()[variable_name] = col_list

Column 'YT_VIDEO_ID': ['SxgaGXcQ8ZY', 'JLwhLr4ZVd0', 'ET8rwerJTg0', 'BfMmAW-58ng', 'VzSq2eW8ZG8']...
Column 'YT_VIDEO_TITLE': ['Krishi Darshan  एकिकृत कृषि प्रणाली', 'Krishi Darshan   Bhagwani ke vikas ka harayana me prayas', 'KRISHI DARSHAN GAON SAMRIDH BHART SAMRIDH', 'Krishi Darshan Chana Fasal', 'Krishi Darshan Zaed Phasal new']...
Column 'YT_VIDEO_LINK': ['https://www.youtube.com/watch?v=SxgaGXcQ8ZY', 'https://www.youtube.com/watch?v=JLwhLr4ZVd0', 'https://www.youtube.com/watch?v=ET8rwerJTg0', 'https://www.youtube.com/watch?v=BfMmAW-58ng', 'https://www.youtube.com/watch?v=VzSq2eW8ZG8']...
Column 'Duration': [1377.0, 1472.0, 1491.0, 1221.0, 1476.0]...
Column 'Channel': ['Doordarshan National', 'Doordarshan National', 'Doordarshan National', 'Doordarshan National', 'Doordarshan National']...


## DOWNLOAD AUDIO

In [ ]:
#flag check for length
print(len(YT_VIDEO_ID) == len(YT_VIDEO_LINK) == len(YT_VIDEO_TITLE) == len(Duration) == len(Channel))

True


In [ ]:
# %%capture #Comment out if you dont want any outputs
def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID

def YT_VIDEO_ID_generator(VIDEO_LINK:str) -> str:
    """Generator function to generate VIDEO_IDS from Link"""
    return VIDEO_LINK[-11:]

def Download_Audio(video_ID, video_Duration = 1800, channel = "channel_name"):
    import yt_dlp # Import yt_dlp inside the function for multiprocessing

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        # 'outtmpl': f'/content/KrishiDarshan/{channel}/{video_ID}/audio_{video_Duration}.%(ext)s'
        'outtmpl': f'/content/KrishiDarshan/{video_ID}/audio_{video_Duration}.%(ext)s'
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YT_Link_Generator(video_ID)])

[youtube] Extracting URL: https://www.youtube.com/watch?v=L5Wtjxslxr0
[youtube] L5Wtjxslxr0: Downloading webpage


[youtube] L5Wtjxslxr0: Downloading android vr player API JSON
[info] L5Wtjxslxr0: Downloading 1 format(s): 251
[download] Destination: /content/KrishiDarshan/L5Wtjxslxr0/audio_1800.webm
[download] 100% of   24.17MiB in 00:00:00 at 45.49MiB/s  
[ExtractAudio] Destination: /content/KrishiDarshan/L5Wtjxslxr0/audio_1800.mp3
Deleting original file /content/KrishiDarshan/L5Wtjxslxr0/audio_1800.webm (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=mOTuf5FGtl0
[youtube] mOTuf5FGtl0: Downloading webpage


[youtube] mOTuf5FGtl0: Downloading android vr player API JSON
[info] mOTuf5FGtl0: Downloading 1 format(s): 251
[download] Destination: /content/KrishiDarshan/mOTuf5FGtl0/audio_1800.webm
[download] 100% of   20.16MiB in 00:00:00 at 36.96MiB/s  
[ExtractAudio] Destination: /content/KrishiDarshan/mOTuf5FGtl0/audio_1800.mp3
Deleting original file /content/KrishiDarshan/mOTuf5FGtl0/audio_1800.webm (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=eVg608_uTB4
[youtube] eVg608_uTB4: Downloading webpage


[youtube] eVg608_uTB4: Downloading android vr player API JSON
[info] eVg608_uTB4: Downloading 1 format(s): 251
[download] Destination: /content/KrishiDarshan/eVg608_uTB4/audio_1800.webm
[download] 100% of   20.16MiB in 00:00:00 at 38.62MiB/s  
[ExtractAudio] Destination: /content/KrishiDarshan/eVg608_uTB4/audio_1800.mp3
Deleting original file /content/KrishiDarshan/eVg608_uTB4/audio_1800.webm (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=oI_MqLnUPbQ
[youtube] oI_MqLnUPbQ: Downloading webpage


[youtube] oI_MqLnUPbQ: Downloading android vr player API JSON
[info] oI_MqLnUPbQ: Downloading 1 format(s): 251
[download] Destination: /content/KrishiDarshan/oI_MqLnUPbQ/audio_1800.webm
[download] 100% of   20.58MiB in 00:00:00 at 29.77MiB/s  
[ExtractAudio] Destination: /content/KrishiDarshan/oI_MqLnUPbQ/audio_1800.mp3
Deleting original file /content/KrishiDarshan/oI_MqLnUPbQ/audio_1800.webm (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=ET8rwerJTg0
[youtube] ET8rwerJTg0: Downloading webpage


...

[youtube] 5WlTrocLyL4: Downloading android vr player API JSON
[info] 5WlTrocLyL4: Downloading 1 format(s): 140
[download] /content/KrishiDarshan/5WlTrocLyL4/audio_1800.m4a has already been downloaded
[download] 100% of   25.47MiB
[ExtractAudio] Destination: /content/KrishiDarshan/5WlTrocLyL4/audio_1800.mp3
Deleting original file /content/KrishiDarshan/5WlTrocLyL4/audio_1800.m4a (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=zuPiN5oul7k
[youtube] zuPiN5oul7k: Downloading webpage


[youtube] zuPiN5oul7k: Downloading android vr player API JSON
[info] zuPiN5oul7k: Downloading 1 format(s): 251
[download] Destination: /content/KrishiDarshan/zuPiN5oul7k/audio_1800.webm
[download] 100% of   20.02MiB in 00:00:00 at 36.02MiB/s  
[ExtractAudio] Destination: /content/KrishiDarshan/zuPiN5oul7k/audio_1800.mp3
Deleting original file /content/KrishiDarshan/zuPiN5oul7k/audio_1800.webm (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=zDIE5aprOFQ
[youtube] zDIE5aprOFQ: Downloading webpage


[youtube] zDIE5aprOFQ: Downloading android vr player API JSON
[info] zDIE5aprOFQ: Downloading 1 format(s): 251
[download] Destination: /content/KrishiDarshan/zDIE5aprOFQ/audio_1800.webm
[download] 100% of   22.62MiB in 00:00:00 at 41.77MiB/s  
[ExtractAudio] Destination: /content/KrishiDarshan/zDIE5aprOFQ/audio_1800.mp3
Deleting original file /content/KrishiDarshan/zDIE5aprOFQ/audio_1800.webm (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=OlGB8Zk7ohQ
[youtube] OlGB8Zk7ohQ: Downloading webpage


[youtube] OlGB8Zk7ohQ: Downloading android vr player API JSON
[info] OlGB8Zk7ohQ: Downloading 1 format(s): 251
[download] Destination: /content/KrishiDarshan/OlGB8Zk7ohQ/audio_1800.webm
[download] 100% of   19.15MiB in 00:00:00 at 21.04MiB/s  
[ExtractAudio] Destination: /content/KrishiDarshan/OlGB8Zk7ohQ/audio_1800.mp3
Deleting original file /content/KrishiDarshan/OlGB8Zk7ohQ/audio_1800.webm (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=0CDTXSlDSns
[youtube] 0CDTXSlDSns: Downloading webpage


[youtube] 0CDTXSlDSns: Downloading android vr player API JSON
[info] 0CDTXSlDSns: Downloading 1 format(s): 251
[download] Destination: /content/KrishiDarshan/0CDTXSlDSns/audio_1800.webm
[download] 100% of   18.28MiB in 00:00:00 at 29.29MiB/s  
[ExtractAudio] Destination: /content/KrishiDarshan/0CDTXSlDSns/audio_1800.mp3
Deleting original file /content/KrishiDarshan/0CDTXSlDSns/audio_1800.webm (pass -k to keep)


In [ ]:
import multiprocessing as mp

inputs = list(zip(YT_VIDEO_ID, Duration, Channel))
print(len(inputs))

# Defined task : Don Eladio

if __name__ == '__main__':
    process_num = mp.cpu_count()
    print("TOTAL CORES: ", process_num)

    with mp.Pool(processes= process_num) as pool:
        # Remove 5 limit here if u want to download complete list
        pool.starmap(Download_Audio, inputs[:5])


    print("JOB FINISHED")


12416
TOTAL CORES:  2
[youtube] Extracting URL: https://www.youtube.com/watch?v=JLwhLr4ZVd0
[youtube] JLwhLr4ZVd0: Downloading webpage
[youtube] Extracting URL: https://www.youtube.com/watch?v=SxgaGXcQ8ZY
[youtube] SxgaGXcQ8ZY: Downloading webpage


[youtube] SxgaGXcQ8ZY: Downloading android vr player API JSON


[youtube] JLwhLr4ZVd0: Downloading android vr player API JSON
[info] SxgaGXcQ8ZY: Downloading 1 format(s): 140
[info] JLwhLr4ZVd0: Downloading 1 format(s): 140
[download] Destination: /content/KrishiDarshan/Doordarshan National/JLwhLr4ZVd0/audio_1472.0.m4a
[download]   1.1% of   22.29MiB at    4.64MiB/s ETA 00:04[download] Destination: /content/KrishiDarshan/Doordarshan National/SxgaGXcQ8ZY/audio_1377.0.m4a
[download] 100% of   22.29MiB in 00:00:00 at 36.11MiB/s  
[download] 100% of   20.85MiB in 00:00:01 at 18.03MiB/s  
[FixupM4a] Correcting container of "/content/KrishiDarshan/Doordarshan National/JLwhLr4ZVd0/audio_1472.0.m4a"
[FixupM4a] Correcting container of "/content/KrishiDarshan/Doordarshan National/SxgaGXcQ8ZY/audio_1377.0.m4a"
[youtube] Extracting URL: https://www.youtube.com/watch?v=ET8rwerJTg0
[youtube] ET8rwerJTg0: Downloading webpage
[youtube] Extracting URL: https://www.youtube.com/watch?v=BfMmAW-58ng
[youtube] BfMmAW-58ng: Downloading webpage


[youtube] BfMmAW-58ng: Downloading android vr player API JSON
[info] BfMmAW-58ng: Downloading 1 format(s): 140


[youtube] ET8rwerJTg0: Downloading android vr player API JSON
[info] ET8rwerJTg0: Downloading 1 format(s): 140
[download] Destination: /content/KrishiDarshan/Doordarshan National/ET8rwerJTg0/audio_1491.0.m4a
[download]   0.3% of   23.01MiB at    2.34MiB/s ETA 00:09[download] Destination: /content/KrishiDarshan/Doordarshan National/BfMmAW-58ng/audio_1221.0.m4a
[download] 100% of   23.01MiB in 00:00:00 at 37.61MiB/s  
[FixupM4a] Correcting container of "/content/KrishiDarshan/Doordarshan National/ET8rwerJTg0/audio_1491.0.m4a"
[download]  72.0% of   18.83MiB at   20.67MiB/s ETA 00:00[youtube] Extracting URL: https://www.youtube.com/watch?v=VzSq2eW8ZG8
[youtube] VzSq2eW8ZG8: Downloading webpage
[download] 100% of   18.83MiB in 00:00:01 at 12.37MiB/s  
[FixupM4a] Correcting container of "/content/KrishiDarshan/Doordarshan National/BfMmAW-58ng/audio_1221.0.m4a"


[youtube] VzSq2eW8ZG8: Downloading android vr player API JSON
[info] VzSq2eW8ZG8: Downloading 1 format(s): 140
[download] Destination: /content/KrishiDarshan/Doordarshan National/VzSq2eW8ZG8/audio_1476.0.m4a
[download] 100% of   22.78MiB in 00:00:00 at 42.45MiB/s  
[FixupM4a] Correcting container of "/content/KrishiDarshan/Doordarshan National/VzSq2eW8ZG8/audio_1476.0.m4a"
JOB FINISHED


In [ ]:
#Zipping the files

!zip -r "/content/KrishiDarshan.zip" "/content/drive/MyDrive/AnnamAI Tasks/Krishi Darshan/Dataset"

  adding: content/KrishiDarshan/ (stored 0%)
  adding: content/KrishiDarshan/OlGB8Zk7ohQ/ (stored 0%)
  adding: content/KrishiDarshan/OlGB8Zk7ohQ/audio_1800.mp3 (deflated 3%)
  adding: content/KrishiDarshan/zuPiN5oul7k/ (stored 0%)
  adding: content/KrishiDarshan/zuPiN5oul7k/audio_1800.mp3 (deflated 1%)
  adding: content/KrishiDarshan/ET8rwerJTg0/ (stored 0%)
  adding: content/KrishiDarshan/ET8rwerJTg0/audio_1800.mp3 (deflated 1%)
  adding: content/KrishiDarshan/mOTuf5FGtl0/ (stored 0%)
  adding: content/KrishiDarshan/mOTuf5FGtl0/audio_1800.mp3 (deflated 1%)
  adding: content/KrishiDarshan/oI_MqLnUPbQ/ (stored 0%)
  adding: content/KrishiDarshan/oI_MqLnUPbQ/audio_1800.mp3 (deflated 1%)
  adding: content/KrishiDarshan/zDIE5aprOFQ/ (stored 0%)
  adding: content/KrishiDarshan/zDIE5aprOFQ/audio_1800.mp3 (deflated 3%)
  adding: content/KrishiDarshan/5WlTrocLyL4/ (stored 0%)
  adding: content/KrishiDarshan/5WlTrocLyL4/audio_1800.mp3 (deflated 1%)
  adding: content/KrishiDarshan/0CDTXSlDSns/ 

In [ ]:
import os
import glob

# Define the directory to clean
directory = '/content/KrishiDarshan/'

# Find all .wav files in the directory and its subdirectories
wav_files = glob.glob(directory + '**/*.m4a', recursive=True)

# Delete each .wav file
for file_path in wav_files:
    try:
        os.remove(file_path)
        print(f"Deleted: {file_path}")
    except OSError as e:
        print(f"Error deleting {file_path}: {e}")

print("All .wav files have been removed from the KrishiDarshan directory.")

Deleted: /content/KrishiDarshan/OlGB8Zk7ohQ/audio_1800.m4a
Deleted: /content/KrishiDarshan/zuPiN5oul7k/audio_1800.m4a
Deleted: /content/KrishiDarshan/mOTuf5FGtl0/audio_1800.m4a
Deleted: /content/KrishiDarshan/oI_MqLnUPbQ/audio_1800.m4a
Deleted: /content/KrishiDarshan/zDIE5aprOFQ/audio_1800.m4a
Deleted: /content/KrishiDarshan/0CDTXSlDSns/audio_1800.m4a
Deleted: /content/KrishiDarshan/L5Wtjxslxr0/audio_1800.m4a
Deleted: /content/KrishiDarshan/eVg608_uTB4/audio_1800.m4a
All .wav files have been removed from the KrishiDarshan directory.
